# Personal Finance Assistant Agent 💰

### Gen AI Bootcamp — Session 5
### LangChain Agents & Function Calling | AWS Bedrock

**Objective:**  
Build an AI-powered Personal Finance Assistant using LangChain Agents
and AWS Bedrock. The agent can intelligently select and use financial
tools based on the user's request.

### Tools:
1. Calculate Expense
2. Get Budget Status
3. Convert Currency
4. Calculate Savings Goal
5. Get Spending Tip

In [1]:
# Install required packages

import subprocess
import sys

print("📦 Installing required packages...")

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'langchain>=0.3.25',
    'langchain-aws>=0.2.24',
    'langchain-community>=0.3.23',
    'langchain-core>=0.3.62',
    'boto3>=1.38.0',
    'pydantic>=2.11.4',
    'python-dotenv'
], check=True)

print("✅ All packages installed!")

📦 Installing required packages...
✅ All packages installed!


In [2]:
import os
import math
from datetime import datetime

from langchain_core.tools import tool
from langchain_aws import ChatBedrockConverse
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent

print("✅ Imports completed!")

✅ Imports completed!


In [3]:
# ── AWS Configuration ──────────────────────────────────────

AWS_REGION = 'ap-southeast-2'
MODEL_ID = 'global.amazon.nova-2-lite-v1:0'

# ==========================================================
# ADDING MY AWS BEDROCK API KEY HERE
# ==========================================================

BEDROCK_API_KEY = "PASTE_YOUR_AWS_BEARER_TOKEN_HERE"

# ==========================================================

os.environ['AWS_BEARER_TOKEN_BEDROCK'] = BEDROCK_API_KEY
os.environ['AWS_DEFAULT_REGION'] = AWS_REGION

print("✅ AWS environment configured")
print(f"   Region : {AWS_REGION}")
print(f"   Model  : {MODEL_ID}")

✅ AWS environment configured
   Region : ap-southeast-2
   Model  : global.amazon.nova-2-lite-v1:0


In [4]:
# Initialize AWS Bedrock LLM

print("🤖 Initializing AWS Bedrock LLM...\n")

llm = ChatBedrockConverse(
    model_id=MODEL_ID,
    region_name=AWS_REGION,
    temperature=0.7,
    max_tokens=512,
)

test = llm.invoke([
    HumanMessage(content="Say: Personal Finance Agent ready!")
])

print("✅ LLM initialized successfully!")
print(f"   Model  : {MODEL_ID}")
print(f"   Region : {AWS_REGION}")
print(f"   Test   : {test.content}")

🤖 Initializing AWS Bedrock LLM...

✅ LLM initialized successfully!
   Model  : global.amazon.nova-2-lite-v1:0
   Region : ap-southeast-2
   Test   : ### **Personal Finance Agent Ready!**

Hello! I'm your **Personal Finance Agent**, here to help you manage, plan, and optimize your finances. Whether you're looking to budget smarter, invest wisely, pay off debt, save for a goal, or just understand your financial health better—**I'm ready to assist!**

---

### ✅ **What I Can Help With:**

**1. Budgeting & Expense Tracking**  
   - Create a personalized monthly budget  
   - Identify areas to cut expenses  
   - Set up savings goals  

**2. Debt Management**  
   - Avalanches vs. snowball method  
   - Prioritize high-interest debt  
   - Create a debt payoff plan  

**3. Savings & Emergency Funds**  
   - How much to save and where  
   - High-yield savings account recommendations  
   - Automate your savings  

**4. Investing Guidance**  
   - Beginner-friendly investment options (ETFs, 

### Tool 1 — Calculate Expense 💸

This tool records an expense by taking the amount, category,
and description and returns a formatted expense entry with today's date.

In [5]:
@tool
def calculate_expense(amount: float, category: str, description: str) -> str:
    """
    Log an expense with amount, category and description.

    Args:
        amount: Expense amount in PKR.
        category: Expense category such as food, transport,
                  entertainment, or shopping.
        description: Short description of the expense.

    Returns:
        A formatted expense entry with today's date.
    """

    date = datetime.now().strftime("%Y-%m-%d")

    return (
        f"Expense logged successfully:\n"
        f"- Date: {date}\n"
        f"- Amount: PKR {amount:,.2f}\n"
        f"- Category: {category.title()}\n"
        f"- Description: {description}"
    )

In [6]:
print("🧪 Testing calculate_expense...\n")

result = calculate_expense.invoke({
    "amount": 2000,
    "category": "transport",
    "description": "Ride to university"
})

print(result)

🧪 Testing calculate_expense...

Expense logged successfully:
- Date: 2026-09-24
- Amount: PKR 2,000.00
- Category: Transport
- Description: Ride to university


### Tool 2 — Get Budget Status 📊

This tool checks the remaining monthly budget for a given
spending category using predefined budget limits.

In [7]:
@tool
def get_budget_status(category: str) -> str:
    """
    Check remaining monthly budget for a spending category.

    Args:
        category: Spending category such as food, transport,
                  entertainment, or shopping.

    Returns:
        The monthly budget and remaining amount for the category.
    """

    budgets = {
        "food": 15000,
        "transport": 10000,
        "entertainment": 8000,
        "shopping": 12000
    }

    category = category.lower().strip()

    if category not in budgets:
        return (
            f"Category '{category}' is not available. "
            f"Choose from: food, transport, entertainment, shopping."
        )

    budget = budgets[category]

    # Sample spending values for the assignment
    spending = {
        "food": 6500,
        "transport": 3500,
        "entertainment": 5000,
        "shopping": 7000
    }

    spent = spending[category]
    remaining = budget - spent

    return (
        f"Budget Status — {category.title()}:\n"
        f"- Monthly Budget: PKR {budget:,.0f}\n"
        f"- Amount Spent: PKR {spent:,.0f}\n"
        f"- Remaining Budget: PKR {remaining:,.0f}"
    )

In [8]:
print("🧪 Testing get_budget_status...\n")

result = get_budget_status.invoke({
    "category": "entertainment"
})

print(result)

🧪 Testing get_budget_status...

Budget Status — Entertainment:
- Monthly Budget: PKR 8,000
- Amount Spent: PKR 5,000
- Remaining Budget: PKR 3,000


### Tool 3 — Convert Currency 💱

This tool converts an amount between currencies using predefined
exchange rates.


In [9]:
@tool
def convert_currency(
    amount: float,
    from_currency: str,
    to_currency: str
) -> str:
    """
    Convert between currencies using fixed exchange rates.

    Args:
        amount: Amount to convert.
        from_currency: Original currency, such as USD or PKR.
        to_currency: Target currency, such as PKR or USD.

    Returns:
        Converted currency amount.
    """

    # Fixed rates relative to USD
    rates = {
        "USD": 1.0,
        "PKR": 280.0,
        "EUR": 0.92,
        "GBP": 0.79
    }

    from_currency = from_currency.upper().strip()
    to_currency = to_currency.upper().strip()

    if from_currency not in rates or to_currency not in rates:
        return (
            "Unsupported currency. "
            "Available currencies: USD, PKR, EUR, GBP."
        )

    # Convert source currency → USD → target currency
    amount_in_usd = amount / rates[from_currency]
    converted_amount = amount_in_usd * rates[to_currency]

    return (
        f"{amount:,.2f} {from_currency} = "
        f"{converted_amount:,.2f} {to_currency}"
    )

In [10]:
print("🧪 Testing convert_currency...\n")

result = convert_currency.invoke({
    "amount": 50,
    "from_currency": "USD",
    "to_currency": "PKR"
})

print(result)

🧪 Testing convert_currency...

50.00 USD = 14,000.00 PKR


### Tool 4 — Calculate Savings Goal 🎯

This tool calculates how many months are required to reach
a savings target based on the amount saved each month.

In [11]:
@tool
def calculate_savings_goal(
    target_amount: float,
    monthly_savings: float
) -> str:
    """
    Calculate how long it will take to reach a savings target.

    Args:
        target_amount: Total amount the user wants to save.
        monthly_savings: Amount the user can save each month.

    Returns:
        Number of months required to reach the target.
    """

    if target_amount <= 0:
        return "Target amount must be greater than zero."

    if monthly_savings <= 0:
        return "Monthly savings must be greater than zero."

    months = math.ceil(target_amount / monthly_savings)

    return (
        f"Savings Goal:\n"
        f"- Target Amount: PKR {target_amount:,.0f}\n"
        f"- Monthly Savings: PKR {monthly_savings:,.0f}\n"
        f"- Time Required: {months} month(s)"
    )

In [12]:
print("🧪 Testing calculate_savings_goal...\n")

result = calculate_savings_goal.invoke({
    "target_amount": 100000,
    "monthly_savings": 10000
})

print(result)

🧪 Testing calculate_savings_goal...

Savings Goal:
- Target Amount: PKR 100,000
- Monthly Savings: PKR 10,000
- Time Required: 10 month(s)


### Tool 5 — Get Spending Tip 💡

This tool provides a practical money-saving recommendation
based on the category where the user is overspending.

In [13]:
@tool
def get_spending_tip(category: str) -> str:
    """
    Return a practical money-saving tip for a spending category.

    Args:
        category: Category where the user tends to overspend.

    Returns:
        A practical money-saving tip.
    """

    tips = {
        "food":
            "Try meal planning and set a weekly food budget. "
            "Reducing food delivery orders can also lower spending.",

        "transport":
            "Compare public transport, carpooling, and ride-hailing "
            "costs. Planning trips together can reduce transport expenses.",

        "entertainment":
            "Set a monthly entertainment limit and look for free or "
            "low-cost activities before spending on paid entertainment.",

        "shopping":
            "Use a 24-hour waiting rule before non-essential purchases "
            "and compare prices before buying."
    }

    category = category.lower().strip()

    if category not in tips:
        return (
            f"No specific tip available for '{category}'. "
            f"Available categories: food, transport, entertainment, shopping."
        )

    return f"Money-saving tip for {category.title()}:\n{tips[category]}"

In [14]:
print("🧪 Testing get_spending_tip...\n")

result = get_spending_tip.invoke({
    "category": "food"
})

print(result)

🧪 Testing get_spending_tip...

Money-saving tip for Food:
Try meal planning and set a weekly food budget. Reducing food delivery orders can also lower spending.


# Now registering all five tools

In [15]:
# Registering all Personal Finance tools

tools = [
    calculate_expense,
    get_budget_status,
    convert_currency,
    calculate_savings_goal,
    get_spending_tip
]

print("📦 Finance tools registered:\n")

for i, t in enumerate(tools, 1):
    print(f"{i}. {t.name}")

📦 Finance tools registered:

1. calculate_expense
2. get_budget_status
3. convert_currency
4. calculate_savings_goal
5. get_spending_tip


In [16]:
# Creating Personal Finance Assistant Agent

print("🧠 Creating Personal Finance Assistant...\n")

finance_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a helpful Personal Finance Assistant. "
        "Help users with basic expense logging, budget checking, "
        "currency conversion, savings goals, and spending tips. "
        "Use the available tools whenever they are relevant. "
        "For requests involving multiple financial tasks, use all "
        "necessary tools. "
        "Present results clearly and concisely. "
        "Do not invent financial data that is not provided by the tools."
    )
)

print("✅ Personal Finance Agent created successfully!")


🧠 Creating Personal Finance Assistant...

✅ Personal Finance Agent created successfully!


In [18]:
# Agent runner

def run_agent(query: str) -> str:
    """
    Send a user query to the Personal Finance Agent
    and return the final response.
    """

    result = finance_agent.invoke({
        'messages': [
            {
                'role': 'user',
                'content': query
            }
        ]
    })

    return result['messages'][-1].content


print("✅ Agent runner ready!")
print('Call: run_agent("your financial question here")')

✅ Agent runner ready!
Call: run_agent("your financial question here")


# REQUIRED QUERY 1

In [19]:
print("=" * 70)
print("💰 QUERY 1 — CURRENCY CONVERSION + EXPENSE")
print("=" * 70)

query = "I spent 50 USD on food today, convert it to PKR and log it"

print(f"\n👤 User Query:\n{query}\n")
print("-" * 70)

try:
    answer = run_agent(query)
    print(f"\n🤖 Agent Response:\n{answer}")

except Exception as e:
    print(f"⚠️ Error: {e}")

💰 QUERY 1 — CURRENCY CONVERSION + EXPENSE

👤 User Query:
I spent 50 USD on food today, convert it to PKR and log it

----------------------------------------------------------------------

🤖 Agent Response:
Great! I've converted your $50 USD food expense to PKR and logged it for you.

**Conversion Result:**
- $50.00 USD = PKR 14,000.00

**Expense Logged:**
- Date: 2026-09-24
- Amount: PKR 14,000.00
- Category: Food
- Description: Spent 50 USD on food

Your food expense for today has been successfully recorded in your expense tracker.


# REQUIRED QUERY 2

In [20]:
print("=" * 70)
print("📊 QUERY 2 — BUDGET STATUS")
print("=" * 70)

query = "What is my remaining budget for entertainment?"

print(f"\n👤 User Query:\n{query}\n")
print("-" * 70)

try:
    answer = run_agent(query)
    print(f"\n🤖 Agent Response:\n{answer}")

except Exception as e:
    print(f"⚠️ Error: {e}")

📊 QUERY 2 — BUDGET STATUS

👤 User Query:
What is my remaining budget for entertainment?

----------------------------------------------------------------------

🤖 Agent Response:
Your remaining budget for entertainment this month is **PKR 3,000**. 

Here's the breakdown:
- **Monthly Entertainment Budget:** PKR 8,000  
- **Amount Already Spent:** PKR 5,000  
- **Remaining Balance:** PKR 3,000  

You have PKR 3,000 left to spend on entertainment this month.


# REQUIRED QUERY 3

In [21]:
print("=" * 70)
print("🎯 QUERY 3 — SAVINGS GOAL")
print("=" * 70)

query = (
    "I want to save 100,000 PKR. "
    "I can save 10,000 per month. "
    "When will I reach my goal?"
)

print(f"\n👤 User Query:\n{query}\n")
print("-" * 70)

try:
    answer = run_agent(query)
    print(f"\n🤖 Agent Response:\n{answer}")

except Exception as e:
    print(f"⚠️ Error: {e}")

🎯 QUERY 3 — SAVINGS GOAL

👤 User Query:
I want to save 100,000 PKR. I can save 10,000 per month. When will I reach my goal?

----------------------------------------------------------------------

🤖 Agent Response:
You will reach your savings goal of **PKR 100,000** in **10 months** if you save **PKR 10,000** each month. Keep up the consistent saving habit!


# REQUIRED QUERY 4

In [22]:
print("=" * 70)
print("💡 QUERY 4 — SPENDING TIP")
print("=" * 70)

query = (
    "I keep overspending on food, "
    "give me a money-saving tip"
)

print(f"\n👤 User Query:\n{query}\n")
print("-" * 70)

try:
    answer = run_agent(query)
    print(f"\n🤖 Agent Response:\n{answer}")

except Exception as e:
    print(f"⚠️ Error: {e}")

💡 QUERY 4 — SPENDING TIP

👤 User Query:
I keep overspending on food, give me a money-saving tip

----------------------------------------------------------------------

🤖 Agent Response:
Here's a money-saving tip for your food expenses:

**Try meal planning and set a weekly food budget.** This helps you buy only what you need and avoid impulse purchases. Also, reducing food delivery orders can significantly lower your spending.

Would you like help setting up a specific food budget or need tips on meal planning?


# REQUIRED QUERY 5

In [23]:
print("=" * 70)
print("🚗 QUERY 5 — EXPENSE + BUDGET")
print("=" * 70)

query = (
    "Log 2000 PKR for transport "
    "and show my transport budget status"
)

print(f"\n👤 User Query:\n{query}\n")
print("-" * 70)

try:
    answer = run_agent(query)
    print(f"\n🤖 Agent Response:\n{answer}")

except Exception as e:
    print(f"⚠️ Error: {e}")

🚗 QUERY 5 — EXPENSE + BUDGET

👤 User Query:
Log 2000 PKR for transport and show my transport budget status

----------------------------------------------------------------------

🤖 Agent Response:
I've logged your transport expense and checked your budget status.

**Expense Logged:**
- Date: 2026-09-24
- Amount: PKR 2,000.00
- Category: Transport
- Description: Transport expense

**Transport Budget Status:**
- Monthly Budget: PKR 10,000
- Amount Spent: PKR 3,500
- Remaining Budget: PKR 6,500

You still have PKR 6,500 remaining in your transport budget for this month.


# MY COMPOUND QUERY — MULTIPLE TOOLS

In [24]:
print("=" * 70)
print("🌟 MY COMPOUND QUERY — MULTIPLE TOOLS")
print("=" * 70)

query = (
    "I want to save 60,000 PKR by spending less on food. "
    "Give me a food-saving tip and tell me how many months "
    "it would take if I save 10,000 PKR per month."
)

print(f"\n👤 User Query:\n{query}\n")
print("-" * 70)

try:
    answer = run_agent(query)
    print(f"\n🤖 Agent Response:\n{answer}")

except Exception as e:
    print(f"⚠️ Error: {e}")

🌟 MY COMPOUND QUERY — MULTIPLE TOOLS

👤 User Query:
I want to save 60,000 PKR by spending less on food. Give me a food-saving tip and tell me how many months it would take if I save 10,000 PKR per month.

----------------------------------------------------------------------

🤖 Agent Response:
Here's your food-saving advice and savings timeline:

**Food-Saving Tip:**
Try meal planning and set a weekly food budget. Reducing food delivery orders can also lower spending.

**Savings Goal Calculation:**
- Target Amount: PKR 60,000
- Monthly Savings: PKR 10,000
- Time Required: 6 months

With consistent monthly savings of PKR 10,000, you'll reach your target of PKR 60,000 in 6 months.


# Interactive chatbot

In [25]:
# Interactive Personal Finance Assistant

print("=" * 70)
print("💰 PERSONAL FINANCE ASSISTANT")
print("=" * 70)

print("""
Hello! I am your Personal Finance Assistant.

You can ask me about:
- Expenses
- Budgets
- Currency conversion
- Savings goals
- Spending tips

Type 'quit' to exit.
""")

while True:

    user_input = input("You: ").strip()

    if not user_input:
        continue

    if user_input.lower() in ["quit", "exit", "bye"]:
        print("\nFinance Assistant: Goodbye! 💰")
        break

    try:
        answer = run_agent(user_input)
        print(f"\nFinance Assistant: {answer}\n")

    except Exception as e:
        print(f"\n⚠️ Error: {e}\n")

💰 PERSONAL FINANCE ASSISTANT

Hello! I am your Personal Finance Assistant.

You can ask me about:
- Expenses
- Budgets
- Currency conversion
- Savings goals
- Spending tips

Type 'quit' to exit.

You: I want to save 150000 PKR and can save 15000 PKR every month. How long will it take?

Finance Assistant: Based on your savings plan:

- **Target Amount:** PKR 150,000
- **Monthly Savings:** PKR 15,000
- **Time Required:** **10 months**

You'll be able to reach your savings goal in approximately 10 months by saving PKR 15,000 each month.

You: I spend too much on shopping. Give me a money-saving tip.

Finance Assistant: Here's a practical money-saving tip for shopping:

**Use a 24-hour waiting rule** before making non-essential purchases. This helps you avoid impulse buying. Also, **compare prices** across different stores and online platforms before buying anything to ensure you're getting the best deal.

You: I spent 25 USD on shopping today. Convert it to PKR, log the expense, and tell 